In [ ]:
from llama_index.llms.ollama import Ollama
import re
import json
import pandas as pd
import sys
import os
import numpy as np
import spacy
from heapq import nlargest
from llama_index.core import PromptTemplate
sys.path.append(os.path.abspath("MHQA-ontology"))
from prompts.prompt_templates import *
from rank_bm25 import BM25Okapi
from sklearn.metrics.pairwise import cosine_similarity
from typing import List, Dict, Tuple, Union
from tqdm import tqdm

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
llm = Ollama(model="llama3.3", request_timeout= 280.0,keep_alive=-1)
response = llm.complete("What are the recurring archetypes in Bulgarian political myths?")
# print(response)

In [ ]:
Hotpot_train = json.load(open('dataset/hotpot/hotpot_train_v1.1.json'))
# Hotpot_train = json.load(open('MHQA-ontology/dataset/hotpot/hotpot_train_v1.1.json'))

In [ ]:
import importlib
import utils.RAG_process
importlib.reload(utils.RAG_process)
from utils.RAG_process import rag_generate

dataset = Hotpot_train[:10]

RAG_context = rag_generate(dataset,llm)


In [ ]:
RAG_context

In [ ]:
dataset_df = pd.DataFrame(dataset)
refine_dataset = dataset_df[dataset_df['question'].isin(RAG_context['Question'])].reset_index(drop=True)
refine_dataset

In [ ]:
import importlib
import utils.RAG_process
importlib.reload(utils.RAG_process)
from utils.RAG_process import refine_supporting_facts
# Example usage:

# Process with verbose output
refined_df = refine_supporting_facts(RAG_context, RAG_context, refine_dataset)

# Check results for a specific row
idx = 1
print(f"Original facts: {RAG_context['Retrieval Facts'][idx]}")
print(f"Refined facts: {refined_df['Retrieval Facts'][idx]}")


In [ ]:
import ast

def compare_facts(refined_df):
    """
    Compares if all Supporting Facts are in Retrieval Facts.
    Sets coverage_all to 1 if all are present, else 0.
    """
    df = refined_df.copy()
    
    # Function to check if all supporting facts are in retrieval facts
    def check_coverage(row):
        supporting = row['Supporting Facts']
        retrieval = row['Retrieval Facts']
        
        # Convert strings to lists if needed
        if isinstance(supporting, str):
            supporting = ast.literal_eval(supporting)
        if isinstance(retrieval, str):
            retrieval = ast.literal_eval(retrieval)
        
        # Convert lists to strings for simple comparison
        supporting_str = [str(item) for item in supporting]
        retrieval_str = [str(item) for item in retrieval]
        
        # Check if all supporting facts are in retrieval facts
        all_present = all(item in retrieval_str for item in supporting_str)
        return 1 if all_present else 0
    
    # Apply the function to each row
    df['coverage_all'] = df.apply(check_coverage, axis=1)
    
    return df

refined_df2 = compare_facts(refined_df)


In [ ]:
refined_df2

In [ ]:
RAG_context_1 = refined_df2[refined_df2['coverage_all'] == 1].reset_index(drop=True)
RAG_context_1

In [ ]:
from utils.KG_builder import build_rdflib_knowledge_graph,process_contexts_in_batches

index = 1
all_facts = RAG_context_1['Retrieval Result'][index]

question = RAG_context_1['Question'][index]
context_data = RAG_context_1['Retrieval Result'][index]




knowledge_graph = process_contexts_in_batches(
    context_items=all_facts,
    llm=llm,
    prompt_template=prompt_template_context,
    batch_size=2  # Process 2 contexts at a time
)

kg = build_rdflib_knowledge_graph(knowledge_graph)

In [ ]:
import importlib
import ultis.iterative_reasoner
importlib.reload(ultis.iterative_reasoner)
from ultis.iterative_reasoner import process_question_with_kg_awareness



# Process the question with KG-aware exploration
result = process_question_with_kg_awareness(
    kg=kg,  
    context_data=context_data, 
    question=question,
    llm=llm,
    max_iterations=4  
)

# Display the concise answer
print(f"Question: {result['question']}")
print(f"Answer: {result['answer']}")  # Now returns just "Delhi" instead of a sentence
print(f"Evidence source: {result['evidence_source']}")

In [ ]:
result['answer']    

In [ ]:
result['full_response_for_final']    

SELECT ?head_office_in 
WHERE { 
    ?company kg:name "The Oberoi Group" .
    ?company kg:is_a "hotel_company" .
    ?company kg:has_head_office_in ?head_office_in .
}

In [ ]:
query2 = """
PREFIX kg: <http://example.org/kg/>

SELECT ?s
WHERE {
    ?s kg:name "Richard Nixon" .
}
"""
#   ?o kg:name "Milhouse" .
#   ?s kg:created_by ?o .
#   ?o kg:named_after ?named_after .
# Execute the query
for row in kg.query(query2):
    print(row)

'question': 'Musician and satirist Allie Goertz wrote a song about the "The Simpsons" character Milhouse, who Matt Groening named after who?',

['Allie Goertz',
   ['Allison Beth "Allie" Goertz (born March 2, 1991) is an American musician.',
    ' Goertz is known for her satirical songs based on various pop culture topics.',
    ' Her videos are posted on YouTube under the name of Cossbysweater.',

['Milhouse Van Houten',
   ['Milhouse Mussolini van Houten is a fictional character featured in the animated television series "The Simpsons", voiced by Pamela Hayden, and created by Matt Groening who named the character after President Richard Nixon\'s middle name.',


In [ ]:
Hotpot_train[2]